# Architectural Design: Multilayer Perceptron

In this notebook, we will break down the structural components of our MLP. We begin with the most fundamental building block: the Dense Layer.

## Task 1: The Atomic Layer (Initialization)

A dense (fully connected) layer is responsible for maintaining its own set of parameters: **Weights ($W$)** and **Biases ($b$)**.

### Why do we initialize weights with small random values?
**Symmetry Breaking:** If we initialized all weights to zero, every neuron in the hidden layer would compute the exact same output, receive the exact same gradient during backpropagation, and update in the exact same way. They would remain identical. By initializing with small random values, we break this symmetry, allowing each neuron to learn different features from the data.

**Why small values?** Large initial weights can cause activation functions (like Sigmoid) to saturate immediately (produce values extremely close to 0 or 1), making their gradients close to zero and causing the network to stop learning (the vanishing gradient problem).

In [ ]:
import numpy as np

class DenseLayer:
    def __init__(self, input_size: int, output_size: int, random_seed: int = 42):
        """
        Initializes weights and biases for the Dense layer.
        
        Args:
            input_size: Number of input features/neurons from the previous layer.
            output_size: Number of neurons in this current layer.
            random_seed: Used to ensure reproducibility for the human evaluator.
            
        Why:
        - Weights are initialized with a standard normal distribution scaled by 0.1 
          to keep the values small and prevent early activation saturation.
        - Biases are initialized to zeros because symmetry breaking is already 
          handled by the weights.
        """
        np.random.seed(random_seed)

        # Weight matrix shape: (input_size, output_size)
        # We scale by 0.1 to keep weights small.
        self.weights = np.random.randn(input_size, output_size) * 0.1

        # Bias vector shape: (1, output_size)
        # Broadcasting in NumPy will automatically apply this across the batch size.
        # np.zeros is used to initialize biases to zero.
        self.biases = np.zeros((1, output_size))

    def _repr_html_(self):
        """Helper to neatly print the layer info in Jupyter."""
        return f"<b>DenseLayer</b> (Inputs: {self.weights.shape[0]}, Neurons: {self.weights.shape[1]}) <br> Weights Shape: {self.weights.shape} | Biases Shape: {self.biases.shape}"

### Example Initialization
Let's create a layer that takes **30 input features** (e.g., the Wisconsin breast cancer dataset) and maps them to a hidden layer of **24 neurons**.

In [2]:
# Create the first hidden layer
hidden_layer_1 = DenseLayer(input_size=30, output_size=24)
hidden_layer_1

## Task 2: Layer Forward Pass (Data Flow)

Now we must define how data moves through the layer. The forward pass applies a linear transformation followed by a non-linear activation function.

### The Linear Step
The layer calculates the weighted sum of inputs:
$$ Z = X \cdot W + b $$

**Dimensional Alignment:**
- $X$ (Inputs): Shape `(BatchSize, Input_Features)`
- $W$ (Weights): Shape `(Input_Features, Neurons)`
- $Z$ (Output): Shape `(BatchSize, Neurons)` - Inner dimensions cancel out. 
- $b$ (Biases): Shape `(1, Neurons)` - Broadcasting adds this to every row.

### The Non-Linear Step
To allow the network to learn complex patterns, we apply a non-linear function (like **Sigmoid** or **ReLU**) to the linear output $Z$.

In [3]:
import sys
import os
# Add parent directory to path to import our src modules
sys.path.append(os.path.abspath('..'))
from src.activations import sigmoid, relu

class DenseLayer:
    def __init__(self, input_size: int, output_size: int, activation_name: str = 'sigmoid', random_seed: int = 42):
        np.random.seed(random_seed)
        self.weights = np.random.randn(input_size, output_size) * 0.1
        self.biases = np.zeros((1, output_size))
        self.activation_name = activation_name
        
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """
        Executes the forward pass for this layer.
        
        Why:
        We compute the dot product to combine inputs with their learned importance (weights),
        add the bias to shift the activation threshold, and finally pass the result 
        through a non-linear activation function.
        """
        # 1. Linear Transformation
        self.z = np.dot(inputs, self.weights) + self.biases
        
        # 2. Non-linear Activation
        if self.activation_name == 'sigmoid':
            self.output = sigmoid(self.z)
        elif self.activation_name == 'relu':
            self.output = relu(self.z)
        else:
            self.output = self.z # Linear
            
        return self.output
        
    def _repr_html_(self):
        return f"<b>DenseLayer</b> (Inputs: {self.weights.shape[0]}, Neurons: {self.weights.shape[1]}, Activation: {self.activation_name})"


In [4]:

print("--- Concrete Forward Pass Example ---")
# Create a new layer with the updated class
layer = DenseLayer(input_size=30, output_size=24, activation_name='sigmoid')

# Synthetic batch of 2 patients with 30 features (random data)
np.random.seed(123)
X_batch = np.random.randn(2, 30)

# Execute the forward pass
output = layer.forward(X_batch)

print(f"X_batch shape: {X_batch.shape}")
print(f"Weights shape: {layer.weights.shape}")
print(f"Z and Output shape: {output.shape} -> Expected (2, 24)")

--- Concrete Forward Pass Example ---
X_batch shape: (2, 30)
Weights shape: (30, 24)
Z and Output shape: (2, 24) -> Expected (2, 24)
